In [ ]:
DATA_SPLIT_MODE = "extended"  # options: "summary", "extended"


In [ ]:
import os
import json
import statistics
from collections import defaultdict, Counter
import math

age_trainval = []
age_test = []

def _to_float(x):
    try:
        return float(x)
    except:
        return None

def add_age(dataset_id, age_value):
    ds = str(dataset_id).lower()
    a = _to_float(age_value)
    if a is None:
        return
    if ds.endswith("7225") or ds.endswith("76dd"):
        age_trainval.append(a)
    elif ds.endswith("cc9a"):
        age_test.append(a)

def _rankdata(values):
    n = len(values)
    order = sorted(range(n), key=lambda i: values[i])
    ranks = [0.0] * n
    i = 0
    r = 1
    while i < n:
        j = 1
        while j + 1 < n and values[order[j+1]] == values[order[i]]:
            j +=1
        avg = (r + (r + (j-1))) / 2.0
        for k in range(i, j+1):
            ranks[order[k]] = avg
        r += (j -i +1)
        i = j + 1
    return ranks

def mann_whitney_u_two_sided(x,y):
    n1,n2 = len(x), len(y)
    allv = x + y
    ranks = _rankdata(allv)
    r1 = sum(ranks[:n1])
    u1 = r1 -n1 * (n1 + 1) / 2.0
    u2 = n1 * n2 -u1
    u = min(u1, u2)

    mu = n1 * n2 / 2.0
    sigma = math.sqrt(n1 * n2 *( n1 + n2 + 1) / 12.0)
    if sigma == 0:
        return u, 1.0

    z = (u - mu) / sigma
    p = math.erfc(abs(z) / math.sqrt(2.0))
    return u, p

def print_age_pvalue():
    n1, n2 = len(age_trainval), len(age_test)
    print("=== Age p-value (Train_Val vs Test) ===")
    print("Train_Val n =", n1, "Test n =", n2)



In [ ]:
from scipy.stats import fisher_exact

# counts: [[male_trainval, female_trainval],
#          [male_test,     female_test]]
male_tv = 0
female_tv = 0
male_test = 0
female_test = 0

def add_gender(dataset_id, gender_value):
    global male_tv, female_tv, male_test, female_test

    ds = str(dataset_id).lower()
    g = str(gender_value).strip().lower()

    # unify possible values
    is_male = (g in ["male", "m", "1", "man"])
    is_female = (g in ["female", "f", "0", "woman"])

    # if value is unexpected, skip
    if (not is_male) and (not is_female):
        return

    if ds.endswith("7225") or ds.endswith("76dd"):
        if is_male:
            male_tv += 1
        else:
            female_tv += 1
    elif ds.endswith("cc9a"):
        if is_male:
            male_test += 1
        else:
            female_test += 1

def print_gender_pvalue():
    table = [[male_tv, female_tv],
             [male_test, female_test]]
    oddsratio, p = fisher_exact(table, alternative="two-sided")
    print("=== Gender p-value (Train_Val vs Test) ===")
    print("Contingency table [[male_tv, female_tv],[male_test, female_test]]:", table)
    print("Fisher exact p =", p)


In [ ]:
from scipy.stats import fisher_exact

# --- EVI counts ---
evi_pos_tv = 0
evi_neg_tv = 0
evi_pos_te = 0
evi_neg_te = 0

# --- MFI counts ---
mfi_pos_tv = 0
mfi_neg_tv = 0
mfi_pos_te = 0
mfi_neg_te = 0

def _norm_binary_label(v):
    """
    Return 1 for positive, 0 for negative, None for unknown.
    Adjust mapping if your raw labels differ.
    """
    s = str(v).strip().lower()
    if s in ["1", "yes", "y", "true", "positive", "+", "pos"]:
        return 1
    if s in ["0", "no", "n", "false", "negative", "-", "neg"]:
        return 0
    return None

def add_evi(dataset_id, label_value):
    global evi_pos_tv, evi_neg_tv, evi_pos_te, evi_neg_te
    ds = str(dataset_id).lower()
    lab = _norm_binary_label(label_value)
    if lab is None:
        return

    is_tv = ds.endswith("7225") or ds.endswith("76dd")
    is_te = ds.endswith("cc9a")
    if not (is_tv or is_te):
        return

    if is_tv:
        if lab == 1: evi_pos_tv += 1
        else:        evi_neg_tv += 1
    else:
        if lab == 1: evi_pos_te += 1
        else:        evi_neg_te += 1

def add_mfi(dataset_id, label_value):
    global mfi_pos_tv, mfi_neg_tv, mfi_pos_te, mfi_neg_te
    ds = str(dataset_id).lower()
    lab = _norm_binary_label(label_value)
    if lab is None:
        return

    is_tv = ds.endswith("7225") or ds.endswith("76dd")
    is_te = ds.endswith("cc9a")
    if not (is_tv or is_te):
        return

    if is_tv:
        if lab == 1: mfi_pos_tv += 1
        else:        mfi_neg_tv += 1
    else:
        if lab == 1: mfi_pos_te += 1
        else:        mfi_neg_te += 1


In [ ]:
from collections import Counter
import pandas as pd

crm_missing_reason_counts = Counter()
crm_missing_cases = []
crm_mfi_rows = []

In [ ]:
# Define the root directory where datasets are stored
datasets_dir_path = "/path/to/datasets"

# Get the list of datasets (patient cases)
datasets = os.listdir(datasets_dir_path)

# Dataset to exclude
excluded_dataset = "DATASET_UUID_REDACTED"

# Data storage
overall_clinical_data = defaultdict(Counter)  # Stores overall counts (except age)
dataset_clinical_data = {}  # Stores per-dataset counts
overall_age_list = []  # Stores all ages for computing median & range
dataset_age_data = defaultdict(list)  # Stores age lists per dataset

# Iterate over each dataset (patient case)
for dataset in datasets:
    # Skip the excluded dataset
    if dataset == excluded_dataset:
        continue

    dataset_dir_path = os.path.join(datasets_dir_path, dataset)

    # Load the eForms JSON file
    eforms_file_path = os.path.join(dataset_dir_path, "eforms.json")
    if not os.path.exists(eforms_file_path):
        continue

    with open(eforms_file_path, "r") as f:
        studies = json.load(f)

    # Per-dataset storage
    dataset_counts = defaultdict(Counter)

    # Iterate through all studies in this dataset
    # for study in studies:
    for study_idx, study in enumerate(studies):
        try:
            age = study['eForm']['pages'][0]['page_data']['age_at_diagnosis']['value']
            gender = study['eForm']['pages'][1]['page_data']['gender']['value']
            cT = study['eForm']['pages'][2]['page_data']['tumor_clinical_category']['value']
            cN = study['eForm']['pages'][2]['page_data']['regional_nodes_clinical_category']['value']
            cM = study['eForm']['pages'][2]['page_data']['metastasis_clinical_category']['value']
            EVI_label = study["eForm"]['pages'][2]['page_data']['extramural_invation_mr']['value']
            MFI_label = study["eForm"]['pages'][2]['page_data']['mesorectal_invation_mr']['value']
            
            # CRM_label = study['eForm']['pages'][2]['page_data']['circ_resection_margin']['value']
            # CRM_label = str(CRM_label).strip() if CRM_label is not None and str(CRM_label).strip() else "MISSING"

            CRM_raw = study['eForm']['pages'][2]['page_data']['circ_resection_margin']['value']

            # classify missing reason (simple)
            if CRM_raw is None:
                CRM_label = "MISSING"
                crm_reason = "value_is_none"
            elif isinstance(CRM_raw, str) and CRM_raw.strip() == "":
                CRM_label = "MISSING"
                crm_reason = "value_is_empty_string"
            elif str(CRM_raw).strip().lower() in {"null", "none", "nan", "na", "n/a"}:
                CRM_label = "MISSING"
                crm_reason = f"null_like_string: {str(CRM_raw).strip()}"
            else:
                CRM_label = str(CRM_raw).strip()
                crm_reason = None
            
            # record missing details
            if CRM_label == "MISSING":
                crm_missing_reason_counts[crm_reason] += 1
                crm_missing_cases.append({
                    "dataset": dataset,
                    "study_idx": study_idx,
                    "raw_value": CRM_raw
                })

            # collect MFI-CRM pairs for cross-tab (only valid CRM)
            if CRM_label != "MISSING":
                MFI_clean = str(MFI_label).strip() if MFI_label is not None and str(MFI_label).strip() else "MISSING"
                crm_mfi_rows.append({
                    "dataset": dataset,
                    "study_idx": study_idx,
                    "MFI": MFI_clean,
                    "CRM": CRM_label
                })

            # Store age for median and range calculation
            overall_age_list.append(age)
            dataset_age_data[dataset].append(age)
            add_age(dataset, age)
            add_gender(dataset, gender)
            add_evi(dataset, EVI_label)
            add_mfi(dataset, MFI_label)

            # Store overall counts
            overall_clinical_data['gender'][gender] += 1
            overall_clinical_data['cT'][cT] += 1
            overall_clinical_data['cN'][cN] += 1
            overall_clinical_data['cM'][cM] += 1
            overall_clinical_data['CRM'][CRM_label] += 1

            # Store per-dataset counts
            dataset_counts['gender'][gender] += 1
            dataset_counts['cT'][cT] += 1
            dataset_counts['cN'][cN] += 1
            dataset_counts['cM'][cM] += 1
            dataset_counts['CRM'][CRM_label] += 1

        except KeyError as e:
            crm_missing_reason_counts[f"KeyError: {e}"] += 1
            crm_missing_cases.append({
                "dataset": dataset,
                "study_idx": study_idx,
                "raw_value": "KEYERROR"
            })
            print(f"Missing data in dataset {dataset}, study {study_idx}: {e}")

    # Store dataset-specific statistics
    dataset_clinical_data[dataset] = dataset_counts

# Print overall statistics
print("\n=== Overall Clinical Data Across All Datasets ===")

# Compute and print age statistics
if overall_age_list:
    overall_median_age = statistics.median(overall_age_list)
    overall_age_range = (min(overall_age_list), max(overall_age_list))
    print(f"\nAge at Diagnosis - Median: {overall_median_age}, Range: {overall_age_range}")

for field, counter in overall_clinical_data.items():
    print(f"\n{field}:")
    for value, count in counter.items():
        print(f" {value}: {count} ")

# Print per-dataset statistics
print("\n=== Per-Dataset Clinical Data ===")

for dataset, data in dataset_clinical_data.items():
    print(f"\nDataset: {dataset}")

    # Compute and print per-dataset age statistics
    age_list = dataset_age_data.get(dataset, [])
    if age_list:
        median_age = statistics.median(age_list)
        age_range = (min(age_list), max(age_list))
        print(f" Age at Diagnosis - Median: {median_age}, Range: {age_range}")

    for field, counter in data.items():
        print(f" {field}:")
        for value, count in counter.items():
            print(f" {value}: {count} ")

print("\n=== CRM Missing Reason Summary ===")
for k, v in crm_missing_reason_counts.items():
    print(f"{k}: {v}")

print("\n=== CRM Missing Cases (first 50) ===")
for x in crm_missing_cases[:50]:
    print(f"dataset={x['dataset']}, study_idx={x['study_idx']}, raw_value={repr(x['raw_value'])}")

print(f"\nTotal missing cases recorded for inspection: {len(crm_missing_cases)}")

# Build dataframe
df_ct = pd.DataFrame(crm_mfi_rows).copy()

print("len(crm_mfi_rows) =", len(crm_mfi_rows))
print("df_ct columns =", df_ct.columns.tolist())

# ???????????
if {"dataset", "study_idx"}.issubset(df_ct.columns):
    dup_n = df_ct.duplicated(subset=["dataset", "study_idx"]).sum()
    print("duplicate rows by (dataset, study_idx) =", dup_n)

# ---- Normalize MFI (True/False -> MFI+/MFI-) ----
def map_mfi(x):
    # handle bool
    if x is True:
        return "MFI+"
    if x is False:
        return "MFI-"
    # handle string
    s = str(x).strip().lower()
    if s in {"true", "1", "positive", "pos", "+"}:
        return "MFI+"
    elif s in {"false", "0", "negative", "neg", "-"}:
        return "MFI-"
    else:
        return None

# ---- Normalize CRM verbose labels -> CRM+/CRM- ----
def map_crm(x):
    s = str(x).strip().lower()
    if "uninvolved by tumor" in s:
        return "CRM-"
    elif "involved by tumor" in s:
        return "CRM+"
    else:
        return None

df_ct["MFI_bin"] = df_ct["MFI"].apply(map_mfi)
df_ct["CRM_bin"] = df_ct["CRM"].apply(map_crm)

# ??????????
print("\n=== Unrecognized rows (if any) ===")
unk = df_ct[df_ct["MFI_bin"].isna() | df_ct["CRM_bin"].isna()]
if len(unk) > 0:
    print(unk[["MFI", "CRM"]].drop_duplicates())
else:
    print("None")

# ?????????
df_pm = df_ct.dropna(subset=["MFI_bin", "CRM_bin"]).copy()

print("\nTotal valid rows used for crosstab =", len(df_pm))  # ??? 302?????331?CRM missing=29?

# Cross-tab
ctab = pd.crosstab(df_pm["MFI_bin"], df_pm["CRM_bin"])
ctab = ctab.reindex(index=["MFI+", "MFI-"], columns=["CRM+", "CRM-"], fill_value=0)

print("\n=== Cross-tab: MFI x CRM (valid CRM only) ===")
print(ctab)


In [ ]:
import scipy
print(scipy.__version__)

In [ ]:
from scipy.stats import mannwhitneyu
u, p = mannwhitneyu(age_trainval, age_test, alternative="two-sided")
print("U =", u, "p =", p)

In [ ]:
# print_gender_pvalue()
table = [[male_tv, female_tv],
        [male_test, female_test]]
oddsratio, p = fisher_exact(table, alternative="two-sided")
print("=== Gender p-value (Train_Val vs Test) ===")
print("Contingency table [[male_tv, female_tv],[male_test, female_test]]:", table)
print("Fisher exact p =", p)

In [ ]:
# --- EVI p-value ---
table_evi = [[evi_pos_tv, evi_neg_tv],
             [evi_pos_te, evi_neg_te]]
_, p_evi = fisher_exact(table_evi, alternative="two-sided")
print("EVI table:", table_evi, "p =", p_evi)

# --- MFI p-value ---
table_mfi = [[mfi_pos_tv, mfi_neg_tv],
             [mfi_pos_te, mfi_neg_te]]
_, p_mfi = fisher_exact(table_mfi, alternative="two-sided")
print("MFI table:", table_mfi, "p =", p_mfi)

